# Setup
## Imports

In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.6 MB/s eta 0:00:00


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

from scipy import ndimage
from pycocotools import mask as mask_utils

from tqdm.auto import tqdm

## Paths and Device

In [3]:
DATA_ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026"
)

TEST_IMAGE_DIR = (
    DATA_ROOT
    / "test"
    / "test_images"
)

checkpoint_candidates = list(
    Path("/kaggle/input").rglob(
        "unetplusplus_resnet34_focal_dice.pt"
    )
)

if len(checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "Could not find unetplusplus_resnet34_focal_dice.pt. "
        "Attach the completed training notebook output as an input."
    )

if len(checkpoint_candidates) > 1:
    raise RuntimeError(
        "Found multiple focal+Dice checkpoints: "
        f"{checkpoint_candidates}"
    )

CHECKPOINT_PATH = checkpoint_candidates[0]

OUTPUT_PATH = Path(
    "/kaggle/working/submission.csv"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("Checkpoint:", CHECKPOINT_PATH)
print(
    "Test images:",
    len(list(TEST_IMAGE_DIR.glob("*")))
)

Device: cuda
Checkpoint: /kaggle/input/models/catherinebarbeau/unetplusplus-resnet34-focal-dice/pytorch/default/1/unetplusplus_resnet34_focal_dice.pt
Test images: 180


# Model
## Model Architecture

In [4]:
def make_unetplusplus():
    return smp.UnetPlusPlus(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=1,
        classes=1
    )

## Load Checkpoint

In [5]:
model = make_unetplusplus()

state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

model.load_state_dict(state_dict)

if torch.cuda.device_count() > 1:
    print(
        f"Using {torch.cuda.device_count()} GPUs"
    )

    model = nn.DataParallel(
        model
    )

model = model.to(device)
model.eval()

Using 2 GPUs


DataParallel(
  (module): UnetPlusPlus(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          

# Data
## Test Dataset

In [6]:
class TestDataset(Dataset):
    def __init__(
        self,
        image_dir
    ):
        self.image_paths = sorted(
            path
            for path in image_dir.iterdir()
            if path.is_file()
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(
        self,
        index
    ):
        image_path = self.image_paths[index]

        image = Image.open(
            image_path
        ).convert("L")

        image = np.array(
            image,
            dtype=np.float32
        )

        image = (
            torch.from_numpy(image)
            .unsqueeze(0)
            / 255.0
        )

        return image, image_path.stem

## Test DataLoader

In [7]:
test_dataset = TestDataset(
    TEST_IMAGE_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    f"Test images: "
    f"{len(test_dataset)}"
)

Test images: 180


# Inference
## Inference Utilities

In [8]:
def run_inference(
    model,
    loader,
    device
):
    model.eval()

    predictions = []
    image_ids = []

    with torch.no_grad():
        progress_bar = tqdm(
            loader,
            desc="Inference"
        )

        for images, batch_image_ids in progress_bar:
            images = images.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(images)

            probabilities = torch.sigmoid(
                logits
            )

            probabilities = (
                probabilities
                .cpu()
                .numpy()
            )

            predictions.extend(
                probabilities
            )

            image_ids.extend(
                batch_image_ids
            )

    return predictions, image_ids



## Run Inference

In [9]:
predictions, image_ids = run_inference(
    model=model,
    loader=test_loader,
    device=device
)

print(
    f"Predictions: {len(predictions)}"
)

print(
    f"Image IDs: {len(image_ids)}"
)


Inference:   0%|          | 0/90 [00:00<?, ?it/s]

Predictions: 180
Image IDs: 180


# Postprocessing
## Instance Extraction

In [10]:
def extract_instances(
    probability_map,
    threshold=0.5
):
    binary_mask = (
        probability_map[0] >= threshold
    )

    labeled_mask, num_instances = ndimage.label(
        binary_mask
    )

    instances = []

    for instance_id in range(
        1,
        num_instances + 1
    ):
        instance_mask = (
            labeled_mask == instance_id
        )

        instances.append(
            instance_mask.astype(np.uint8)
        )

    return instances

def extract_instances_hysteresis(
    probability_map,
    low_threshold=0.4,
    high_threshold=0.6,
    min_size=50
):
    probabilities = probability_map[0]

    candidate_mask = (
        probabilities >= low_threshold
    )

    seed_mask = (
        probabilities >= high_threshold
    )

    grown_mask = ndimage.binary_propagation(
        seed_mask,
        mask=candidate_mask
    )

    structure = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    labeled_mask, num_instances = ndimage.label(
        grown_mask,
        structure=structure
    )

    instances = []

    for instance_id in range(
        1,
        num_instances + 1
    ):
        instance_mask = (
            labeled_mask == instance_id
        )

        if instance_mask.sum() < min_size:
            continue

        instances.append(
            instance_mask.astype(np.uint8)
        )

    return instances

## RLE Encoding

In [11]:
def encode_rle(mask):
    rle = mask_utils.encode(
        np.asfortranarray(
            mask.astype(np.uint8)
        )
    )

    counts = rle["counts"]

    if isinstance(counts, bytes):
        counts = counts.decode("utf-8")

    return counts

# Submission
## Build Submission

In [12]:
def build_submission(
    predictions,
    image_ids,
    threshold=0.5
):
    rows = []

    for probability_map, image_id in tqdm(
        zip(predictions, image_ids),
        total=len(predictions),
        desc="Building submission"
    ):
        instances = extract_instances(
            probability_map,
            threshold=threshold
        )

        for instance_index, instance_mask in enumerate(
            instances,
            start=1
        ):
            filament_id = (
                f"{image_id}_{instance_index}"
            )

            segmentation_rle = encode_rle(
                instance_mask
            )

            rows.append(
                {
                    "filament_id": filament_id,
                    "segmentation_rle": segmentation_rle
                }
            )

    return pd.DataFrame(rows)

def build_hysteresis_submission(
    predictions,
    image_ids
):
    rows = []

    for probability_map, image_id in tqdm(
        zip(predictions, image_ids),
        total=len(predictions),
        desc="Building hysteresis submission"
    ):
        instances = extract_instances_hysteresis(
            probability_map
        )

        for instance_index, instance_mask in enumerate(
            instances,
            start=1
        ):
            rows.append(
                {
                    "filament_id": (
                        f"{image_id}_{instance_index}"
                    ),
                    "segmentation_rle": encode_rle(
                        instance_mask
                    )
                }
            )

    return pd.DataFrame(rows)

## Sanity Checks

In [13]:
submission_df = build_hysteresis_submission(
    predictions=predictions,
    image_ids=image_ids
)

print(submission_df.shape)
print(submission_df.head())
print(submission_df.columns.tolist())

print(
    "Unique filament IDs:",
    submission_df["filament_id"].is_unique
)

print(
    "Missing RLEs:",
    submission_df["segmentation_rle"].isna().sum()
)

Building hysteresis submission:   0%|          | 0/180 [00:00<?, ?it/s]

(1627, 2)
          filament_id                                   segmentation_rle
0  20110120105534Ch_1  j_Vi27io110O11O0O20O0001fPNGjn1=QQNEon1;oPNFQo...
1  20110120105534Ch_2  WjRQ22lo13N1O1N101O000O10O10O100O001000000M3M3...
2  20110130110334Ch_1  `_fd22no12M4M4L2M2O1N3N5LO100OQQN\Odn1c0ZQN_Oh...
3  20110214175414Mh_1  Z]f[24ho16K4M3I7N2O0O2O1O0POVOiRNl0Um1WOiRNi0W...
4  20110214175414Mh_2  []Za21no12O0O2O1O00100O100O1O010O010O010O1O001...
['filament_id', 'segmentation_rle']
Unique filament IDs: True
Missing RLEs: 0


## Save Submission

In [14]:
submission_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Saved submission to: {OUTPUT_PATH}"
)

Saved submission to: /kaggle/working/submission.csv
